# 3. Post-processing chain

In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib notebook

In [ ]:
import os
import sys
import glob
import scipy
import numpy as np
import pandas as pd
import scipy as sp
import statsmodels.api as sm
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from scipy.optimize import curve_fit
from matplotlib import patches
from matplotlib.ticker import FuncFormatter
from astropy import units as u
from astropy.coordinates import SkyCoord
from pathlib import Path
from tqdm import tqdm 

# PlatoSim libraries
import platosim.plot       as pt
import platosim.noise      as ns
import platosim.utilities  as ut
import platosim.statistics as st
from platosim.simfile      import SimFile
from platosim.simulation   import Simulation
from platosim.lightcurve   import LightCurve
from platosim.matplotlibrc import setup_paper
setup_paper()

import warnings
warnings.simplefilter("ignore")

In [ ]:
# Define paths used throughout
path = '/lhome/nicholas/software/workdir/mocka'
fdir = '/lhome/nicholas/Nextcloud/paperMOCKA/figures'

---
## Test poly-detrend model comparison
---

In [ ]:
# Availble stars: 10[01-10], 20[01-10], and 30[01-10]) 
starID = f'{1001}'.zfill(9)
G, C, Q = 1, 1, 6

# Load test data
path0 = f'{path}/simulations/test/test_hdf5/'
filename = f'{path0}/{starID}/{starID}_Ncam{G}.{C}_Q{Q}.hdf5'
lc = LightCurve(filename)

# Data (rename for OLS to handle)
df = lc.data()
df.time /= 86400.
df = df.rename(columns={'time':'x', 'flux':'y'})

# Test models
model1 = 'y ~ x'
model2 = 'y ~ x + I(x**2)'
model3 = 'y ~ x + I(x**2) + I(x**3)'
model4 = 'y ~ x + I(x**2) + I(x**3) + I(x**4)'
fit1 = sm.OLS.from_formula(formula=model1, data=df).fit()
fit2 = sm.OLS.from_formula(formula=model2, data=df).fit()
fit3 = sm.OLS.from_formula(formula=model3, data=df).fit()
fit4 = sm.OLS.from_formula(formula=model4, data=df).fit()
# fit3.summary()

In [ ]:
# Show procedure of model comparison
st.plot_modelfit(df, fit1, model1, theme='b')
st.plot_modelfit(df, fit2, model2, theme='g')
st.plot_modelfit(df, fit3, model3, theme='r')
st.plot_modelfit(df, fit4, model4, theme='m')
AIC_j = [fit1.aic, fit2.aic, fit3.aic, fit4.aic]
BIC_j = [fit1.bic, fit2.bic, fit3.bic, fit4.bic]
st.model_selection(AIC_j, BIC_j, method='BIC', show=True);

---
## Test post-processing steps (HDF5)
---

In [ ]:
# Availble stars: 10[01-10], 20[01-10], and 30[01-10]) 
starID = f'{1002}'.zfill(9)
G, C, Q = 4, 2, 6

# Load test data
path0 = f'{path}/simulations/test/test_hdf5'
filename = f'{path0}/{starID}/{starID}_Ncam{G}.{C}_Q{Q}.hdf5'
lc = LightCurve(filename)
lc.star()

In [ ]:
df = lc.detrend(model='poly', replace=True, plot=True)

In [ ]:
df = lc.clip(model='wotan', flux_unit='ppt', sigma_lower=4.5, sigma_upper=4.5, replace=True, plot=True);

---
## Test full reduction on HDF5
---

In [ ]:
star = f'{1}'.zfill(9)
idir = f'{path}/simulations/test/test_clean/{star}'
# idir = f'{path}/simulations/tests/test_hdf5/{star}'

# Load ligth curve object
lcs = LightCurve(idir, 'multi')

# Check sim info
N = len(lcs.files('hdf5')) / 8.
lc = LightCurve(lcs.files('hdf5')[0])

# Simulation table
ds = lc.star()

# Check max amplitude
dp = pd.read_feather(f'{path}/simulations/test/varsource/{star}/varsource_001_parameters.ftr')

# Check predicted NSR camera and mission level
tdur = 3600
noise_jitter         = ut.getJitterNoiseLimitNSR(rms=0.037, tdur=tdur, level='camera')
noise_photon_camera  = ut.getPhotonNoiseLimitNSR(ds.mag, passband='P', ncam=1, tdur=tdur)
noise_photon_mission = ut.getPhotonNoiseLimitNSR(ds.mag, passband='P', ncam=N, tdur=tdur)
noise_background     = ut.getBackgroundNoiseLimitNSR(ds.mag, passband='P', tdur=tdur)
noise_camera  = noise_jitter + noise_photon_camera  + noise_background
noise_mission = noise_jitter + noise_photon_mission + noise_background

print(f'Max ampl    : {dp.Amax_mag[0] * 1.037e6:.0f} ppm')
print(f'NSR camera  : {noise_camera:.0f} ppm')
print(f'NSR mission : {noise_mission:.0f} ppm ({int(N)} N-CAM)')
ds

In [ ]:
fig, ax = lcs.plot_multi(suffix='hdf5', group=False, camera=1, quarter=False, 
                         flux_median=144, alpha=0.1, figsize=(9,5))

In [ ]:
lc = lcs.merge(suffix='hdf5', 
               verbose=True,
               detrend='poly',
               flux_group_mean=True, 
               binsize=0.2, 
               clip=False, #True,
               flux_offset=True, 
               flux_err=True, 
               ofile=f'{idir}/lc_{star}.ftr')

In [ ]:
# Introduce data gaps
df = lc.gaps(f'{path}/input/instrumentGAP.tab', replace=True)
df = df.dropna()
df.time /= 86400

# Load variable template
dm = pd.read_feather(f'{path}/simulations/test/varsource/{star}/varsource_001_pulsations.ftr')
dv = pd.DataFrame()
dv['time'] = df.time
dv['dmag'] = ns.timeSeriesFromFourier(df.time, dm.freq, dm.ampl, dm.phase, power=2.2)
dv['flux'] = (10**(-0.4*dv.dmag) - 1) * 1e6

# Plot final light curve
fig, ax = lc.plot(flux_unit='ppt', median_filter=False)
ax.plot(dv.time, dv.flux, '-', c='orange', lw=1);

In [ ]:
# Check the computation of errorbars
df = lc.data()
time = df.time / 86400
plt.figure(figsize=(9,5))
plt.errorbar(time, df.flux, yerr=df.flux_err, fmt=".", color='k', alpha=0.1)
plt.xlim(time.min(), time.max())
plt.xlabel('Time [days]')
plt.ylabel('Normalised flux')
plt.tight_layout();

In [ ]:
# Compute the residuals
df['flux_res'] = (df.flux - 1)*1e6 - dv.flux

# Compute NSR for star
NSR_res = np.std(df.flux_res)
RMS_res = ut.medianAbsoluteDeviation(df.flux_res)

# NSR from old function
dx = df.copy()
dx.time *= 86400
lc = LightCurve(dx, 'multi')
NSR = lc.getNSR(column='flux_res', influx='ppm', unit='ppm', binhour=1)

# Print to screen
print('NSR measure per 600s')
print(f'Max ampl    : {dp.Amax_mag[0] * 1.037e6:.0f} ppm')
print(f'NSR camera  : {noise_camera:.0f} ppm')
print(f'NSR mission : {noise_mission:.0f} ppm ({N} Ncam)')
print('-------------')
print(f'NSR method  : {NSR:.0f} ppm in 1h')
print(f'NSR measure : {NSR_res:.0f} ppm ({N} Ncam)')
print(f'RMS measure : {RMS_res:.0f} ppm ({N} Ncam)')

In [ ]:
# Regression model of residuals
lc = df.rename(columns={'time':'x', 'flux_res':'y', 'flux_err':'y_err'})
lc['x'] = lc['x'].subtract(lc['x'].min())
model = 'y ~ x'
lsFit = sm.WLS.from_formula(formula=model, data=lc).fit()
lsFit.summary(alpha=0.05)

In [ ]:
# Plot regression model and residuals
st.plot_modelfit(lc, lsFit, model, lsModel='WLS', theme='g', xlab='Time [days]', ylab='Residuals [ppt]')
st.plot_residuals(lc, lsFit, theme='g')
st.plot_standardized_residuals(lc, lsFit, K=2, reg='x', lsModel='WLS')

---
## Optimal SNR to extract modes
---

In [ ]:
path0 = f'{path}/simulations/snr'

### Single cadence

In [ ]:
# dx = ut.plotNoisePeakSNR(cadence=600, quarters=1, N=1000, odir=path)

In [ ]:
fig, ax = ns.plotNoisePeakSNR(path0, cadence=600, quarters=8, fap=2, bins=50, figsize=(7.5, 4))

### Multiple cadences

In [ ]:
# dx = ns.getNoisePeakSNR(path, quarters=1, N=10)

In [ ]:
fig, ax = ns.plotMultiCadenceNoisePeakSNR(path0, fap=1, quarters=8, bins=50, show_snr=True, figsize=(7.5, 4));
ax.set_xlim(3.7, 5.8)
fig.savefig(f'{fdir}/OptimalCriterionSNR.png', bbox_inches='tight', dpi=300)

In [ ]:
# BEST FIT COEFFICIENTS

# FAP 10%
q1 = [4.751441961815984, 4.646419784596318, 4.234842108291144]
q2 = [4.863501527270076, 4.758439876832565, 4.340777615792339]
q3 = [4.923252108914589, 4.8116877878757345, 4.425668886924579]
q4 = [4.9669869569925025, 4.862516083165092, 4.476098647285526]
q5 = [4.98942846598262, 4.886290616227712, 4.506715704129862]
q6 = [5.0359025727844555, 4.922331538388892, 4.531516416386822]
q7 = [5.0385894774387046, 4.9395638259157515, 4.552297875935199]
q8 = [5.069942232036235, 4.960187561639028, 4.5862721956125565]
snr10 = np.array([q1, q2, q3, q4, q5, q6, q7, q8])
fap10 = np.ones(len(snr10)) * 10

# FAP 8%
q1 = [4.782985567116099, 4.686789325252121, 4.281951251102321]
q2 = [4.895219481852064, 4.797305009137667, 4.382295813683609]
q3 = [4.960157216344031, 4.8505563734873895, 4.467593862061666]
q4 = [4.996922409199194, 4.899309677901247, 4.514941922304392]
q5 = [5.025912405861571, 4.922160696933164, 4.547942889544587]
q6 = [5.072337956185754, 4.960296669332852, 4.5713168054674975]
q7 = [5.074169685312954, 4.9769127514469345, 4.595362748873413]
q8 = [5.1075052563009615, 4.998480059206906, 4.627299161617078]
snr8 = np.array([q1, q2, q3, q4, q5, q6, q7, q8])
fap8 = np.ones(len(snr8)) * 8

# FAP 6%
q1 = [4.834635101289733, 4.733820911264426, 4.340556948337373]
q2 = [4.941738917271653, 4.839935653201236, 4.436662034249298]
q3 = [5.006117809189146, 4.894677098281585, 4.521218169277734]
q4 = [5.040488659899626, 4.945979289685511, 4.5674777265613935]
q5 = [5.070121790114455, 4.977623777635532, 4.6036870921918345]
q6 = [5.117065787466482, 5.002103348598984, 4.616189688188275]
q7 = [5.1227680986135695, 5.024557753691917, 4.643274441589813]
q8 = [5.154390107953242, 5.045291305022896, 4.675685972431329]
snr6 = np.array([q1, q2, q3, q4, q5, q6, q7, q8])
fap6 = np.ones(len(snr6)) * 6

# FAP 4%
q1 = [4.904358824041422, 4.797333613937177, 4.400652466303963]
q2 = [4.994616951932046, 4.905837966066807, 4.512459886312335]
q3 = [5.068446546378054, 4.959911492153281, 4.592248437697582]
q4 = [5.1024297376411205, 5.007733184246635, 4.639945987395375]
q5 = [5.130284236545739, 5.042588276634922, 4.671854409278135]
q6 = [5.174070117077145, 5.066248464109928, 4.686850529512095]
q7 = [5.179492690650727, 5.0917850622725815, 4.704846278127562]
q8 = [5.225482286947941, 5.105480181672721, 4.740972635060567]
snr4 = np.array([q1, q2, q3, q4, q5, q6, q7, q8])
fap4 = np.ones(len(snr4)) * 4

# FAP 2%
q1 = [5.007283699079344, 4.916244772827773, 4.522354201004243]
q2 = [5.104045478172652, 5.005239801218056, 4.639942383283823]
q3 = [5.172104314263683, 5.05251450603294, 4.711336690410255]
q4 = [5.205691540958301, 5.110952498708383, 4.759260423528731]
q5 = [5.22888123223425, 5.152169889833671, 4.781219551311802]
q6 = [5.275432973156145, 5.178230970269464, 4.788008866569173]
q7 = [5.280368459154169, 5.197812094139507, 4.814891789620325]
q8 = [5.32784380459164, 5.198939187457133, 4.865028808265933]
snr2 = np.array([q1, q2, q3, q4, q5, q6, q7, q8])
fap2 = np.ones(len(snr2)) * 2

# FAP 1%
q1 = [5.099032363235153, 5.06139030759917, 4.469069906810321]
q2 = [5.198279088967152, 5.118046230099662, 4.584393387223682]
q3 = [5.288894196886051, 5.158302762633607, 4.848945837834034]
q4 = [5.294102193742062, 5.232863346891284, 4.858942920527429]
q5 = [5.311611559980742, 5.250664187575773, 4.895351967621635]
q6 = [5.378917955578637, 5.290193573720642, 4.89644868875318]
q7 = [5.374729898899645, 5.308820691115978, 4.931018218429747]
q8 = [5.411561511678405, 5.299983634039764, 4.979189659061848] 
snr1 = np.array([q1, q2, q3, q4, q5, q6, q7, q8])
fap1 = np.ones(len(snr1))

# FAP 0.5%
q1 = [5.2283584483247, 5.139300654559072, 4.7723868180945725]
q2 = [5.309691458335726, 5.243671361003577, 4.861389406587132]
q3 = [5.382026054815434, 5.225276958087124, 4.981278806131673]
q4 = [5.3840281336088545, 5.325152589372627, 4.967849165036273]
q5 = [5.3963747034303005, 5.36272344031128, 5.014040837359414]
q6 = [5.484636980195333, 5.378662604222594, 5.0068987087046795]
q7 = [5.465981832143509, 5.411319280505803, 5.039615870262087]
q8 = [5.5206948785375545, 5.403739350144865, 5.102280214481962]
snr05 = np.array([q1, q2, q3, q4, q5, q6, q7, q8])
fap05 = np.ones(len(snr05)) * 0.5

# FAP 0.1%
q1 = [5.484254692018193, 5.414928279773168, 5.002450189140223]
q2 = [5.561262525313896, 5.513900092145581, 5.065548026369663]
q3 = [5.684501284753266, 5.437315937529918, 5.222092114364679]
q4 = [5.64988078388191, 5.560201329088244, 5.188106000280937]
q5 = [5.699418225306015, 5.734607285629037, 5.2107785792667665]
q6 = [5.649921701407862, 5.5812832757616855, 5.178590146023707]
q7 = [5.62257027641522, 5.650630154633398, 5.302592393831441]
q8 = [5.751536466369376, 5.574414630455777, 5.329219865659611]
snr01 = np.array([q1, q2, q3, q4, q5, q6, q7, q8])
fap01 = np.ones(len(snr01)) * 0.1

# FAP 0.01%
q1 = [5.772338743430787, 6.202782859409507, 5.560570556244156]
q2 = [6.224323506812625, 5.890896661710462, 5.430606388357388]
q3 = [6.1907820711133175, 5.586034730243456, 5.547232681863193]
q4 = [5.95656403869253, 5.896950573705527, 5.52476432406596]
q5 = [6.003399307288348, 5.954491678220407, 6.145481049538461]
q6 = [6.155342528442318, 5.768367312589713, 5.348088751974616]
q7 = [5.906076056502938, 5.924140531057725, 5.748492337539504]
q8 = [6.104093159194544, 5.766214700306862, 5.64730051498106]
snr001 = np.array([q1, q2, q3, q4, q5, q6, q7, q8])
fap001 = np.ones(len(snr001)) * 0.01

N = len(snr10)
quarter = np.arange(1, N+1, 1)
FAP = [fap10, fap8, fap6, fap4, fap2, fap1, fap05, fap01, fap001]
SNR = [snr10, snr8, snr6, snr4, snr2, snr1, snr05, snr01, snr001]

In [ ]:
# start plotting
col = 2
fig = plt.figure(figsize=(10,9))
ax = fig.add_subplot(1,1,1, projection='3d', computed_zorder=False)
ax.view_init(elev=15, azim=135)

# Fetch and plot all data points
fap0 = []
snr0 = []
for fap, snr in zip(FAP, SNR):
    ax.scatter3D(quarter, fap, snr[:,col], 'o', ec='k', lw=0.5, s=30, alpha=1, zorder=2)
    fap0.append(fap.tolist())
    snr0.append(snr[:,col].tolist())

# Get 3D array for surface
x = np.tile(quarter, len(np.array(FAP)))
y = np.array(fap0).flatten()
z = np.array(snr0).flatten()
 
# Plot best fit surface
def func(xy, a, b, c):
    x, y = xy
    return  a * np.log(x) + b * np.log(y) + c
popt, pcov = curve_fit(func, (x, y), z)
xx = np.linspace(x.min(), x.max(), 1000) 
yy = np.linspace(y.min(), y.max(), 1000) 
X, Y = np.meshgrid(xx, yy) 
Z = func((X, Y), *popt) 
surf = ax.plot_surface(X, Y, Z, cmap='Spectral_r', alpha=1, zorder=1) 

# Colorbar
cbar = plt.colorbar(surf, orientation='vertical', extend='both', 
                   anchor=(-1.2, 0.5), shrink=0.26, aspect=20, ticks=[])

# Labels
ax.set_xlabel(r'Mission quarters, $n_{\rm Q}$', labelpad=10)
ax.set_ylabel('FAP [\%]', labelpad=10)
ax.set_zlabel('S/N', labelpad=10, rotation=90)
ax.set_box_aspect(aspect=None, zoom=0.8)
plt.tight_layout(pad=0)

# Save figure and thereafter crop with Linux command "convert":
fig.savefig(f'{fdir}/OptimalCriterionSurface.png', bbox_inches='tight', dpi=300)
os.system(f'convert {fdir}/OptimalCriterionSurface.png -trim {fdir}/OptimalCriterionSurface.png')

# Fit coefficients
print(popt)
print(np.sqrt(np.diag(pcov)))
print(np.linalg.cond(pcov))

In [ ]:
# Cadence used in paper
def y(x, y, c=25):
    if   c == 25:  c1, c2, c3 = 0.13217934, -0.15429918, 5.12996448
    elif c == 50:  c1, c2, c3 = 0.10930811, -0.14909435, 5.05757044   
    elif c == 600: c1, c2, c3 = 0.15992845, -0.16496221, 4.62691121
    return c1*np.log(x) + c2*np.log(y) + c3
print(y(8, 0.1, c=25), y(8, 0.1, c=50), y(8, 0.1, c=600))

---
## Test frequency extraction with STARSHADOW
---

With a PLATOnium conda environment activated, first install software using
```
pip install git+https://github.com/LucIJspeert/star_shadow
```
Open the file needed to compile the `numba` code
```
$CONDA_PREFIX/lib/python3.9/site-packages/star_shadow/run_first_use.py
```
Replace the line:
```
data_dir = script_dir.replace('star_shadow', 'data') 
```
with 
```
data_dir = os.path.join(script_dir, 'data')
```
Check that the `data` folder exists within the directory `$CONDA_PREFIX/lib/python3.9/site-packages/star_shadow`. If not, create this directory and download and place the two files `sim_000_lc.dat` and `mpl_stylesheet.dat` into this folder.

Compile the code with:
```
python $CONDA_PREFIX/lib/python3.9/site-packages/star_shadow/run_first_use.py
```
We further implement a small piece of code that records which modes are abvoe a certain SNR during the prewhitening procedure. Then we can compare this to the default procedure of STAR SHADOW.

Now the code can be used.

In [ ]:
import star_shadow as ss

In [ ]:
# Open model light curve to test with
filename = f'{path}/simulations/tests/varsource/000000001/varsource_001.txt'
df = pd.read_csv(filename, sep=' ', names=['time', 'mag'])

# Convert to format that STAR SHADOW needs
df.time /= 86400.
df['flux'] = ut.fromMagToFlux(df.mag)
df.flux = (df.flux - df.flux.mean())
df['flux_err'] = np.ones_like(df.flux)
df = df.drop(columns=['mag'])
df = df.loc[::24]  # 600s / 25s-> 24 exp
df.to_csv(f'{path}/varsource_600s.dat', sep=' ', index=False, header=False)

In [ ]:
# Test run of analysis script
ss.analyse_lc_from_file(f'{path}/varsource_600s.dat', save_dir=path, stage='freq', overwrite=True, verbose=True)

---
## Plots for paper
---

### Single light curve

In [ ]:
# Load star ID 1003
path0 = f'{path}/simulations/test/test_hdf5'
filename = f'{path0}/000001003/000001003_Ncam1.1_Q1.hdf5'
lc = LightCurve(filename)

In [ ]:
# Plot detrending
df = lc.detrend(model='poly', replace=False, plot=False)
fig, ax = lc.plot_detrend(df, column='flux', plot_oc=False, figsize=(14,8))
fig.savefig(f'{fdir}/pipeline_detrending.png', bbox_inches='tight', dpi=200)

In [ ]:
# Plot outlier rejection
df = lc.clip(column='flux_detrend', model='wotan', flux_unit='ppt', sigma_lower=4.5, sigma_upper=4.5, replace=False)
fig, ax = lc.plot_clip(df, column='flux_detrend', flux_unit='ppt', plot_oc=False, figsize=(14,8))
fig.savefig(f'{fdir}/pipeline_clip.png', bbox_inches='tight', dpi=200)

### Multi-camera light curve

In [ ]:
# Load all light curves for multi-camera light curves
star = f'{1004}'.zfill(9)
idir = f'{path}/simulations/test/test_hdf5/{star}'
lcs = LightCurve(idir, 'multi')

In [ ]:
# Plot multi camera and quarter light curve
fig, ax = lcs.plot_multi(suffix='hdf5', group=False, camera=False, quarter=False, 
                         flux_median=False, alpha=0.1, figsize=(14,10))
# fig.savefig(f'{fdir}/pipeline_multi_camera.png', bbox_inches='tight', dpi=200)

In [ ]:
# Combine observations to single reduced light curve
lc = lcs.merge(suffix='hdf5', 
               verbose=True,
               detrend='poly',
               flux_group_mean=True, 
               clip=False, 
               binsize=0.2, 
               flux_offset=True, 
               flux_err=True, 
               ofile=f'{idir}/lc_{star}.ftr')

In [ ]:
# Introduce data gaps
df = lc.gaps(f'{path}/input/instrumentGAP.tab', replace=True)
df = df.dropna()
df.time /= 86400

# Load variable template
dm = pd.read_feather(f'{path}/simulations/test/varsource/{star}/varsource_001_pulsations.ftr')
dv = pd.DataFrame()
dv['time'] = df.time
dv['dmag'] = ns.timeSeriesFromFourier(df.time, dm.freq, dm.ampl, dm.phase, power=2.2)
dv['flux'] = (10**(-0.4*dv.dmag) - 1) * 1e6

# Plot final light curve
fig, ax = lc.plot(flux_unit='ppm', median_filter=False, figsize=(14, 5))
ax.plot(dv.time, dv.flux, '-', c='orange', lw=0.3, label='Input model')
ax.legend(ncol=2)
fig.savefig(f'{fdir}/pipeline_multi_final.png', bbox_inches='tight', dpi=200)

In [ ]:
# Fetch simulation table
path0 = f'/lhome/nicholas/software/workdir/mocka/simulations/mocka/GDOR'
idir  = f'{path0}/affogato'
vdir  = f'{path0}/varsource'
starID = f'{1002}'.zfill(9)

# Fetch final ligth curve
lc = LightCurve(f'{idir}/final/lc_{starID}.ftr', mode="final")
df = lc.data()

# Compute power spectrum
time = df.time / 86400     # [day]
flux = (df.flux - 1) * 1e6 # [ppm]
freq0, ampl0 = ns.astropyLombScargle(time, flux, 
                                     f0=0, 
                                     fn=3.5, 
                                     df=np.diff(time)[0] * 0.01, 
                                     norm='amplitude')

# Load sim table
dt = pd.read_feather(f'{idir}/table/table_{starID}.ftr')

# Create varsource from pulsations
dx = pd.read_feather(f'{vdir}/pulsations/pulsations_{starID}_001.ftr')
dv = pd.DataFrame()
dv['time'] = df.time / 86400
dv['dmag'] = ns.timeSeriesFromFourier(dv.time, dx.freq, dx.ampl, dx.phase, power=2.2)
dv['flux'] = (10**(-0.4*dv.dmag) - 1) * 1e6

In [ ]:
# Load star results
dm = pd.read_feather(f'{idir}/modes/modes_{starID}.ftr')
dp = pd.read_feather(f'{vdir}/parameters/parameters_{starID}_001.ftr')
df = pd.read_feather(f'{vdir}/pulsations/pulsations_{starID}_001.ftr')

# Correct for gamma factor
df.ampl /= 2.2

# Convert dmag to ppm
df.ampl = (1 - ut.fromMagToFlux(df.ampl)) * 1e6

# Fetch input frequencies in pettern
f_i = 1 / np.array([dp.DeltaP0_day * ((1 + dp.slope)**i - 1)/dp.slope + dp.P0_day for i in range(dp.N_modes[0])])

# Get pattern
dex_df = np.array([ut.findNearestIndex(df.freq, f_i[i]) for i in range(dp.N_modes[0])])
dex_dm = np.array([ut.findNearestIndex(dm.freq, f_i[i]) for i in range(dp.N_modes[0])])
df0 = df.loc[dex_df].reset_index(drop=True)
dm0 = dm.loc[dex_dm].reset_index(drop=True)

# O-C plot
f_oc = df0.freq.to_numpy() - dm0.freq.to_numpy()
A_oc = df0.ampl.to_numpy() - dm0.ampl.to_numpy()
dm1 = dm[dm.passed_snr]

# Remove matches above 0.01 c/d in the OC diagram
x = 0.0005
# x = 0.005
dex = np.where((np.abs(f_oc) > x))[0]
dm0 = dm0.drop(index=dex)
f0_oc = np.delete(f_oc, dex)
A0_oc = np.delete(A_oc, dex)

print(f'Stellar magnitude  : {dp.Pmag[0]:.4f} mag')
print(f'Number of modes    : {dm0.shape[0]}/{dp.N_modes[0]}')
print(f'Limiting amplitude : {dm.ampl.min():.4f} ppm')
print(f'Dominant amplitude : {df.ampl.max():.4f} ppm')
print(f'RMS O-C amplitude  : {ut.rootMeanSquare(A0_oc):.4f} ppm')
print(f'RMS O-C frequency  : {ut.rootMeanSquare(f0_oc)*1e6:.4f} ppm/d')

In [ ]:
# EXAMPLE PLOT OF MODE EXTRACTION
fig, ax = plt.subplots(5, 1, figsize=(14,15))

# Full range
xmin = np.min([df0.freq.min(), dm.freq.min()])
xmax = np.max([df0.freq.max(), dm.freq.max()])
xlim0 = pt.getAxesMinMax(x=[xmin, xmax], percentage=1)
ymin = np.min([df0.ampl.min(), dm.ampl.min()])
ymax = np.max([df0.ampl.max(), dm.ampl.max()])
ylim0 = pt.getAxesMinMax(y=[ymin, ymax], percentage=1)

# Zoomin on noise
xlim1 = (0.002, 0.38)

# Zoom-in on mode pattern
xlim2 = pt.getAxesMinMax(x=df.freq.to_numpy(), percentage=5)[::-1]
ylim2 = pt.getAxesMinMax(x=df.ampl.to_numpy(), percentage=5)

# Plot all modes detected
ax[0].plot(freq0, ampl0, 'k-', lw=0.3)
ax[0].axvspan(0,        xlim1[1], color='b', alpha=0.15, lw=0, label='Zoom-in on noise')
ax[0].axvspan(xlim2[0], xlim2[1], color='g', alpha=0.15, lw=0, label='Zoom-in on pattern')
ax[0].axhline(y=dm.ampl.min(), xmin=xlim0[1], xmax=xlim0[0], ls=':', c='k', label='Detection limit') 
ax[0].plot(df0.freq, df0.ampl, 'o', ms=7, c='orange', mec='k', label='Pattern input')
ax[0].errorbar(dm.freq,  dm.ampl,  xerr=dm.freq_err,  yerr=dm.ampl_err,  fmt=".", ms=12, 
               mec='lightgray', color="orangered",  label='Detected BIC')
ax[0].errorbar(dm1.freq, dm1.ampl, xerr=dm1.freq_err, yerr=dm1.ampl_err, fmt=".", ms=12, 
               mec='lightgray', color="green", label='Detected SNR')
handles, labels = ax[0].get_legend_handles_labels()
order = [0,1,2,3,4,5]
ax[0].legend([handles[i] for i in order], [labels[i] for i in order], 
             loc='upper center', ncol=3, bbox_to_anchor=(0.46, 1.45), fontsize=15.2)
ax[0].set_ylabel(r'$A$ [ppm]')
ax[0].set_yscale('log')
ax[0].set_xlim(xlim0)
ax[0].set_ylim(0.5, 40)

# Zoom-in on noise
ax[1].plot(freq0, ampl0, 'k-', lw=0.8)
ax[1].axvspan(xlim1[0], xlim1[1], color='b', alpha=0.1, lw=0)
ax[1].axhline(y=dm.ampl.min(), xmin=xlim0[1], xmax=xlim0[0], ls=':', c='k')
ax[1].axvline(x=1/(1*(ut.quarter()+2)), ls='--', c='royalblue', lw=1.5, label=r'$1 \times t_{\rm Q}$')
ax[1].axvline(x=1/(2*(ut.quarter()+2)), ls='-.', c='royalblue', lw=1.5, label=r'$2 \times t_{\rm Q}$')
ax[1].axvline(x=1/(3*(ut.quarter()+2)), ls=':',  c='royalblue', lw=1.5, label=r'$3 \times t_{\rm Q}$')
ax[1].axvline(x=1/3,                    ls='--', c='orange',    lw=1.5, label=r'$1 \times t_{\rm wheel}$')
ax[1].axvline(x=1/6,                    ls='-.', c='orange',    lw=1.5, label=r'$2 \times t_{\rm wheel}$')
ax[1].axvline(x=1/12,                   ls=':',  c='orange',    lw=2.0, label=r'$4 \times t_{\rm wheel}$')
ax[1].axvline(x=1/(ut.quarter()/3),     ls='--', c='m',         lw=1.5, label=r'$1 \times t_{\rm mask}$')
ax[1].axvline(x=1/(ut.quarter()/3*2),   ls='-.', c='m',         lw=1.5, label=r'$2 \times t_{\rm mask}$')
ax[1].axvline(x=1/(4*(ut.quarter()+2)), ls='--', c='c',         lw=1.5, label=r'$1 \times t_{\rm yr}$')
ax[1].errorbar(dm.freq, dm.ampl, xerr=dm.freq_err, yerr=dm.ampl_err,  fmt=".", ms=13, 
               mec='lightgray', color="orangered",  label='Detected all')
ax[1].errorbar(dm1.freq, dm1.ampl, xerr=dm1.freq_err, yerr=dm1.ampl_err, fmt=".", ms=13, 
               mec='lightgray', color="green", label='Detected SNR')
handles, labels = ax[1].get_legend_handles_labels()
order = [0,1,2,8,6,7,3,4,5]
ax[1].legend([handles[i] for i in order], [labels[i] for i in order],
             loc='upper center', ncol=3, bbox_to_anchor=(0.75, 1.02), fontsize=13)
ax[1].set_ylabel(r'$A$ [ppm]')
ax[1].set_xscale('log')
ax[1].set_xlim(xlim1)
ax[1].set_ylim(0, 10)

# Zoom-in of pattern
ax[2].plot(freq0, ampl0, 'k-', lw=0.8)
ax[2].axvspan(xlim2[1], xlim2[0], color='g', alpha=0.1, lw=0)
ax[2].axhline(y=dm.ampl.min(), xmin=xlim0[0], xmax=xlim0[1], ls=':', c='k')
ax[2].plot(df0.freq, df0.ampl, 'o', ms=9, c='orange', mec='k')
ax[2].errorbar(dm.freq, dm.ampl, xerr=dm.freq_err, yerr=dm.ampl_err, fmt=".", ms=14, 
               mec='lightgray', color="orangered")
ax[2].errorbar(dm1.freq, dm1.ampl, xerr=dm1.freq_err, yerr=dm1.ampl_err, fmt=".", ms=14, 
               mec='lightgray', color="green")
ax[2].set_ylabel(r'$A$ [ppm]')
ax[2].set_xlim(xlim2[1], xlim2[0])
ax[2].set_ylim(0, 40)

# OC diagram for amplitudes
ax[3].plot([xlim2[0], xlim2[1]], [0, 0], 'k--', alpha=0.5)
ax[3].errorbar(dm0.freq, A0_oc, xerr=None, yerr=dm0.ampl_err, fmt=".", ms=15, 
               mec='lightgray', color="deeppink")
ax[3].set_ylabel(r'O-C ($A$) [ppm]')
ax[3].set_xlim(xlim2[1], xlim2[0])

# OC diagram for frequencies
ax[4].plot([xlim2[0], xlim2[1]], [0, 0], 'k--', alpha=0.5)
ax[4].errorbar(dm0.freq, f0_oc*1e6, xerr=None, yerr=dm0.freq_err*1e6, fmt=".", ms=15, 
               mec='lightgray', color="darkcyan")
ax[4].set_xlabel(r'Frequency, $\nu$ [d$^{-1}$]')
ax[4].set_ylabel(r'O-C ($\nu$) [ppm d$^{-1}$]')
ax[4].set_xlim(xlim2[1], xlim2[0])

# Settings
for i in range(5): ax[i].get_yaxis().set_label_coords(-0.05, 0.5)
plt.tight_layout(h_pad=0.1);
fig.savefig(f'{fdir}/pipeline_frequency.png', bbox_inches='tight', dpi=200)